In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")

# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")


2025-06-16 20:29:26.604537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750120167.128470  110726 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750120167.254567  110726 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750120168.316275  110726 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750120168.316316  110726 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750120168.316318  110726 computation_placer.cc:177] computation placer alr

'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01
Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


In [2]:
print("Creando input estructurado...")
df_train['structured_text'] = df_train.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)

Creando input estructurado...


In [3]:
from sklearn.model_selection import KFold
df_model = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_model = df_model[['text', 'label']].copy()
df_model['label'] = df_model['label'].apply(lambda x: int(x) - 1)


# --- 2. Modelo, Tokenizador y Métricas ---
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# La métrica oficial del reto es F1-Macro
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, predictions, average="macro")}


# --- 3. El Loop de K-Fold ---
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # Usar random_state para resultados reproducibles
all_scores = []

print(f"Iniciando entrenamiento con K-Fold de {N_SPLITS} splits...")

for fold, (train_idx, val_idx) in enumerate(kf.split(df_model)):
    print(f"\n===== FOLD {fold+1}/{N_SPLITS} =====")
    
    # Crear datasets para este fold específico
    train_df = df_model.iloc[train_idx]
    eval_df = df_model.iloc[val_idx]
    
    train_dataset = Dataset.from_pandas(train_df)
    eval_dataset = Dataset.from_pandas(eval_df)

    tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
    tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

    # Directorio de salida único para cada fold
    output_dir = f"./kfold_model_fold_{fold+1}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        
        # --- PARÁMETROS FIJOS (LOS QUE ELEGIMOS) ---
        learning_rate=5e-5, # El valor por defecto, lo hacemos explícito
        num_train_epochs=3,
        per_device_train_batch_size=16,
        warmup_steps=500,
        weight_decay=0.01,
        
        # --- Argumentos de logística ---
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        report_to="none",
    )

    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    score = eval_results["eval_f1_macro"]
    all_scores.append(score)
    print(f"F1-Macro para el Fold {fold+1}: {score:.4f}")

    # Guardamos el modelo final del fold
    trainer.save_model(output_dir)

# --- 4. Resultados Finales de la Validación ---
print("\n===== Resultados K-Fold =====")
print(f"F1-Macro scores por fold: { [round(s, 4) for s in all_scores] }")
print(f"F1-Macro Promedio: {np.mean(all_scores):.4f}")
print(f"Desviación Estándar: {np.std(all_scores):.4f}")


Iniciando entrenamiento con K-Fold de 5 splits...

===== FOLD 1/5 =====


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_110726/1844451329.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Macro
1,1.304700,1.021303,0.546691
2,1.003100,0.988441,0.540145
3,0.735900,1.058128,0.574008


...

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_110726/1844451329.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Macro
1,1.305700,1.073188,0.481679
2,1.025400,0.991242,0.545735
3,0.744000,1.063653,0.561584


F1-Macro para el Fold 5: 0.5616

===== Resultados K-Fold =====
F1-Macro scores por fold: [0.574, 0.5464, 0.581, 0.5405, 0.5616]
F1-Macro Promedio: 0.5607
Desviación Estándar: 0.0155


In [5]:
from scipy.special import softmax
# --- 2. Preprocesamiento del Texto (¡Debe ser IDÉNTICO al de entrenamiento!) ---
print("Preprocesando texto de test...")
df_test['text'] = df_test.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)
test_texts = df_test['text'].tolist()


# --- 3. Cargar Modelos y Realizar Predicciones en Bucle ---
model_paths = [f"./kfold_model_fold_{i+1}" for i in range(N_SPLITS)]
all_probabilities = []

# Cargamos el tokenizador (es el mismo para todos los modelos)
tokenizer = AutoTokenizer.from_pretrained(model_paths[0])

# Tokenizamos los datos de test una sola vez
tokenized_test_dataset = tokenizer(test_texts, padding="max_length", truncation=True, max_length=512, return_tensors="pt")
test_dataset = Dataset.from_dict(tokenized_test_dataset)

print(f"\nIniciando predicción con ensamble de {N_SPLITS} modelos...")
for i, path in enumerate(model_paths):
    print(f"--- Usando modelo del Fold {i+1}/{N_SPLITS} desde '{path}' ---")
    
    # Cargar el modelo del fold actual
    model = AutoModelForSequenceClassification.from_pretrained(path)
    # Si tienes GPU, esto acelera enormemente el proceso
    if torch.cuda.is_available():
        model.to("cuda")

    # Creamos un Trainer simple solo para la predicción
    trainer = Trainer(model=model)
    
    # Obtenemos los logits (salidas crudas del modelo)
    raw_predictions = trainer.predict(test_dataset)
    
    # Convertimos los logits a probabilidades usando la función softmax
    # Esto nos da la "confianza" del modelo en cada una de las 5 clases
    probabilities = softmax(raw_predictions.predictions, axis=1)
    all_probabilities.append(probabilities)
    
    # Liberar memoria (opcional pero buena práctica)
    del model
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- 4. Promediar Probabilidades y Obtener Predicción Final ---
print("\nPromediando las predicciones de todos los modelos...")
# Hacemos la media de las probabilidades obtenidas de los 5 modelos
# El resultado es una matriz de [2500 reseñas x 5 clases]
average_probabilities = np.mean(all_probabilities, axis=0)

# La predicción final para cada reseña es la clase (0 a 4) con la probabilidad promedio más alta
final_predictions_indices = np.argmax(average_probabilities, axis=1)

# Convertimos los índices (0-4) a las etiquetas del reto (1-5)
final_labels = final_predictions_indices + 1


# --- 5. Crear y Guardar el Archivo de Entrega ---
submission_df = pd.DataFrame({'ID': df_test['ID'], 'Polarity': final_labels})

Preprocesando texto de test...

Iniciando predicción con ensamble de 5 modelos...
--- Usando modelo del Fold 1/5 desde './kfold_model_fold_1' ---


--- Usando modelo del Fold 2/5 desde './kfold_model_fold_2' ---


--- Usando modelo del Fold 3/5 desde './kfold_model_fold_3' ---


--- Usando modelo del Fold 4/5 desde './kfold_model_fold_4' ---


--- Usando modelo del Fold 5/5 desde './kfold_model_fold_5' ---



Promediando las predicciones de todos los modelos...


In [6]:
submission_df

,ID,Polarity
0,0,3
1,1,3
2,2,2
3,3,5
4,4,4
...,...,...
2495,2495,4
2496,2496,3
2497,2497,3
2498,2498,3


In [7]:
submission_path = 'submission_final_kfold.csv'
submission_df.to_csv(submission_path, index=False)

In [5]:
# Paso 2: Preparar el DataFrame y convertirlo a un Dataset de Hugging Face
# Renombramos las columnas y ajustamos las etiquetas (si no lo has hecho en el df original)
df_train_beto = df_train.rename(columns={'structured_text': 'text', 'Polarity': 'label'})
df_train_beto['label'] = df_train_beto['label'].apply(lambda x: int(x) - 1)

# Convertir a Dataset
dataset = Dataset.from_pandas(df_train_beto)

# Dividir en entrenamiento y validación
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']


# Paso 3: Cargar el tokenizador y el modelo BETO
# ¡Este es el cambio principal!
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5) # 5 clases de polaridad

# Paso 4: Tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Paso 5: Definir la métrica de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "f1_weighted": f1}

# Paso 6: Configurar y ejecutar el entrenamiento
training_args = TrainingArguments(
    output_dir="./results_beto",     # Nuevo directorio para no sobreescribir el anterior
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_beto',       # Nuevo directorio de logs
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    compute_metrics=compute_metrics,
)

# ¡Iniciar el entrenamiento!
trainer.train()

print("¡Entrenamiento con BETO completado!")

# Para guardar el modelo final y el tokenizador
# trainer.save_model("./mi_modelo_beto_final")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,1.129800,1.124463,0.488000,0.453598
2,1.030600,1.029660,0.558000,0.528876
3,0.660700,1.066399,0.586000,0.589876


¡Entrenamiento con BETO completado!
